# 3PL Workforce Management System: Future Leave Demand Forecasting

**Project:** Capstone 3PL Logistics Workforce Management
**Module:** Predictive Analytics - Model 2: Future Leave Demand
**Frameworks:** Scikit-Learn, Pandas, NumPy

---
## 1. Problem Definition & Objectives
Multi-variable forecasting of future leave request volume over a specified horizon combining day-of-week seasonality, holiday proximity, headcount, role mix, workload, and recent leave velocity.

In [ ]:
import math
import numpy as np
import pandas as pd
from datetime import date, timedelta
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

print('Libraries imported.')

## 2 & 3. Dataset Construction & Model Comparison

In [ ]:
np.random.seed(42)
n_days = 365
base_date = date(2025, 1, 1)
records = []
for i in range(n_days):
    cur_d = base_date + timedelta(days=i)
    dow = cur_d.weekday()
    is_weekend = 1 if dow in (5, 6) else 0
    is_shoulder = 1 if dow in (0, 4) else 0
    is_holiday_near = 1 if (cur_d.month, cur_d.day) in [(1,1), (4,9), (5,1), (6,12), (8,25), (11,1), (11,30), (12,25), (12,30)] else 0
    active_staff = np.random.randint(90, 115)
    avg_ot = np.random.uniform(10, 35)
    lambda_leaves = (active_staff * 0.02) * (1.0 + 0.4*is_shoulder + 0.8*is_holiday_near + 0.02*(avg_ot - 15) - 0.3*is_weekend)
    leaves = np.random.poisson(max(0.1, lambda_leaves))
    records.append({
        'date': cur_d, 'day_of_week': dow, 'is_shoulder': is_shoulder,
        'is_weekend': is_weekend, 'is_holiday_near': is_holiday_near,
        'active_staff': active_staff, 'avg_overtime': avg_ot, 'leave_requests': leaves,
    })
df_leaves = pd.DataFrame(records)
X = df_leaves[['day_of_week', 'is_shoulder', 'is_weekend', 'is_holiday_near', 'active_staff', 'avg_overtime']]
y = df_leaves['leave_requests']
split = int(len(df_leaves) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

models = {
    'Ridge Regression': Ridge(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=80, random_state=42),
}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = math.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f'{name} -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.3f}')

## 4. 14-Day Prediction Horizon & 95% Confidence Interval

In [ ]:
best_reg = models['Gradient Boosting Regressor']
horizon_X = X_test.iloc[:14]
horizon_preds = best_reg.predict(horizon_X)
total_pred = np.sum(horizon_preds)
std_err = np.std(horizon_preds) * math.sqrt(14)
lower_ci = max(0, int(round(total_pred - 1.96 * std_err)))
upper_ci = int(round(total_pred + 1.96 * std_err))
print(f'14-Day Forecast: {int(round(total_pred))} leave requests')
print(f'95% Prediction Interval: {lower_ci} to {upper_ci} requests')

## 5. Model Export & Academic Disclaimer
**Academic Disclaimer:** Forecasted leave requests represent a statistically estimated demand window for workforce scheduling. Unforeseen personal emergencies may alter realized attendance.

In [ ]:
joblib.dump(best_reg, 'leave_demand_gbr.joblib')
print('Leave demand model exported successfully.')